In [ ]:
import os
import math
import json
import pickle
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import logging

logger = logging.getLogger(name=__name__)

repo_root = Path.cwd()
while not (repo_root / "src").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
sys.path.append(str(repo_root))

import src.auxFunctions as auxFunctions
import src.inputMechanicalParametersModel2 as MechanicalParams2
import src.vertexModel2 as vertexModel2
import src.auxFunctionsExpansion as auxFunctionsExpansion


In [ ]:
def run_expansion_simulation(
    cellmap_start,
    geom,
    energyContributions_model,
    # Features to include in expansion
    enable_detachment=False,
    enable_divisions=False,
    enable_collapses=False,
    # Directory for saving 
    output_dir="expansion_simulation",
    # How long to expand the tissue for
    total_steps=15000,
    steps_per_cycle=500,
    # Selecting amount of apoptosis
    collapse_fraction_per_remodel=0.0005,
    relax_after_collapse=10,
    # Selecting threshold for division
    edge_sum_threshold=7,
    division_distance=0.01,
    relax_after_division=10,
    # Selecting line tension relaxation
    tension_decrease_factor=0.4,
    # Selecting the fraction of edges to relax
    relax_tension_fraction=0.5,
    # Selecting factors for change in prefered area and area elasticity 
    pressure_increase_factor=8,
    area_elasticity_value=25,
    # Specifying capsule remodelling to allow expansion
    capsule_tension=300,
    capsule_viscosity=10000,
    # Limit cell cycle events for the inside of the tissue, to avoid boundary effects
    boundary_layers=5,
    # Plotting
    xlim=(-30, 70),
    ylim=(-30, 70),
):
    """
    Run tissue expansion simulation with optional:
    - enable_detachment = ECM's preferred length is reset to each edge's actual length at the start of expansion
    - enable_divisions = allowing cell proliferation if treshold is reached
    - enable_collapses = allowing apoptosis in the model through cell contraction
    - relax_tension_fraction = if <1 causes spatially heterogeneous relaxing of edges
    """

    os.makedirs(output_dir, exist_ok=True)
    json_path = os.path.join(output_dir, "simulation_stats.json")

    # Building simulation description
    features = []
    if enable_detachment:
        features.append("detachment")
    if enable_divisions:
        features.append(f"divisions_thr{edge_sum_threshold}")
    if enable_collapses:
        features.append(f"collapses_{collapse_fraction_per_remodel:.4f}")
    
    if relax_tension_fraction > 0:
        features.append(f"relax_{int(relax_tension_fraction*100)}%_edges")
    
    simulation_desc = " + ".join(features) if features else "basic"

    cellmap = cellmap_start.copy()

    # Creating columns in vert_df to track vertex division 

    v = cellmap.vert_df
    if "new_vert_id" not in v.columns:
        v["new_vert_id"] = np.arange(len(v), dtype=int)
    if "parent_vert_id" not in v.columns:
        v["parent_vert_id"] = np.nan
    if "birth_step" not in v.columns:
        v["birth_step"] = 0
    if "divided_step" not in v.columns:
        v["divided_step"] = np.nan
    next_new_vert_id = int(v["new_vert_id"].max()) + 1

    # Changing model boundary (immitating capsule remodelling)

    boundary_edges, boundary_faces, inside_edges, outside_edges, \
        inside_faces, inside_vertices, outside_vertices = \
        auxFunctions.identify_boundary_layers(cellmap, 1)

    cellmap.vert_df.loc[outside_vertices, "viscosity"] = capsule_viscosity
    cellmap.edge_df.loc[outside_edges, "line_tension"] = capsule_tension

    # Choosing random edges for tension relaxation (relax_tension_fraction)

    edf = cellmap.edge_df
    n_edges = len(edf)
    k = int(np.floor(relax_tension_fraction * n_edges))
    k = max(0, min(k, n_edges))

    if k > 0:
        chosen_edges = np.random.choice(edf.index.to_numpy(), size=k, replace=False)
        edf.loc[chosen_edges, "line_tension"] *= tension_decrease_factor
        print(f"Relaxed {k}/{n_edges} edges to {tension_decrease_factor:.2f}× original tension")

    # Setting preffered area and area elasticity

    cellmap.face_df["prefered_area"] *= pressure_increase_factor
    cellmap.face_df["area_elasticity"] = area_elasticity_value

    # ECM detachment, setting edge's prefered length to its current length 
    if enable_detachment:
        cellmap.edge_df["prefered_length"] = cellmap.edge_df["length"]

    # Creating tracking lists
    division_stats = []
    collapse_stats = []
    remodel_nverts_stats = []
    collapse_target_stats = []
    inside_edges_total_stats = []

    # Saving initial state
    print(f"\n{'='*60}")
    print(f"Starting simulation: {simulation_desc}")
    print(f"Output directory: {output_dir}")
    print(f"{'='*60}\n")

    stats_data = {
        "simulation_description": simulation_desc,
        "parameters": {
            "total_steps": total_steps,
            "steps_per_cycle": steps_per_cycle,
            "collapse_fraction_per_remodel": collapse_fraction_per_remodel,
            "edge_sum_threshold": edge_sum_threshold,
            "division_distance": division_distance,
            "relax_after_collapse": relax_after_collapse,
            "relax_after_division": relax_after_division,
            "tension_decrease_factor": tension_decrease_factor,
            "relax_tension_fraction": relax_tension_fraction,
            "pressure_increase_factor": pressure_increase_factor,
            "area_elasticity_value": area_elasticity_value,
            "capsule_tension": capsule_tension,
            "capsule_viscosity": capsule_viscosity,
            "boundary_layers": boundary_layers,
        },
        "enabled_features": {
            "detachment": enable_detachment,
            "divisions": enable_divisions,
            "collapses": enable_collapses,
        },
    }

    with open(json_path, "w") as f:
        json.dump(stats_data, f, indent=2)

    # Saving initial checkpoint
    with open(os.path.join(output_dir, "checkpoint_000000.pkl"), "wb") as f:
        pickle.dump(cellmap, f)

    # Saving initial visualization
    fig, ax = auxFunctions.view(cellmap, geom, show_axes=True, xlim=xlim, ylim=ylim)
    plt.title(f"Step 0 | {simulation_desc}", fontsize=10)
    plt.savefig(os.path.join(output_dir, "progress_000000.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)

    # Helper functions

    def relax(cellmap, nsteps: int):
        """Run mechanical relaxation without topological changes."""
        if nsteps <= 0:
            return cellmap
        energyContributions_model.compute_energy(cellmap)
        cellmap_out, _, _, _, _ = vertexModel2.solveEuler(
            cellmap, geom, energyContributions_model, int(nsteps)
        )
        return cellmap_out

    def ensure_new_vert_ids(cellmap, next_id):
        """Assign unique IDs to any new vertices."""
        vdf = cellmap.vert_df
        if "new_vert_id" not in vdf.columns:
            vdf["new_vert_id"] = np.nan
        missing = vdf["new_vert_id"].isna()
        if missing.any():
            n = int(missing.sum())
            vdf.loc[missing, "new_vert_id"] = np.arange(next_id, next_id + n, dtype=int)
            next_id += n
        return next_id

    # Main simulation loop

    for step in range(0, total_steps, steps_per_cycle):
        current_step = step + steps_per_cycle
        print(f"\n--- Cycle {current_step}/{total_steps} ---")

        # Re-identifying model boundary
        boundary_edges, boundary_faces, inside_edges, outside_edges, \
            inside_faces, inside_vertices, outside_vertices = \
            auxFunctions.identify_boundary_layers(cellmap, boundary_layers)

        current_divisions = 0
        current_collapses = 0

        # Carrying out edge collapses (apoptosis)

        if enable_collapses:
            valid_inside_edges = [e for e in inside_edges if e in cellmap.edge_df.index]
            n_inside = len(valid_inside_edges)
            target_collapse = int(math.floor(collapse_fraction_per_remodel * n_inside))
            target_collapse = max(0, min(target_collapse, n_inside))

            inside_edges_total_stats.append(n_inside)
            collapse_target_stats.append(target_collapse)

            if target_collapse > 0:
                edges_to_collapse = np.random.choice(valid_inside_edges, size=target_collapse, replace=False)
                for edge in edges_to_collapse:
                    try:
                        cellmap = auxFunctionsExpansion.collapse_single_edge_expansion(
                            cellmap, geom, energyContributions_model, edge
                        )
                        current_collapses += 1
                    except Exception as e:
                        print(f"  Warning: collapse failed for edge {edge}: {e}")

            print(f"  Collapses: {current_collapses}/{target_collapse}")

            cellmap.reset_index()
            cellmap.reset_topo()
            next_new_vert_id = ensure_new_vert_ids(cellmap, next_new_vert_id)

            if relax_after_collapse > 0:
                cellmap = relax(cellmap, relax_after_collapse)
                cellmap.reset_index()
                cellmap.reset_topo()

        # Re-identifying boundaries after collapses
        boundary_edges, boundary_faces, inside_edges, outside_edges, \
            inside_faces, inside_vertices, outside_vertices = \
            auxFunctions.identify_boundary_layers(cellmap, boundary_layers)

        # Carrying out division (splitting vertices connected to edge length > threshold)

        if enable_divisions:
            # Finding vertices to divide using edge_sum_threshold
            vertices_to_divide = []
            for v in inside_vertices:
                if v not in cellmap.vert_df.index:
                    continue
                # Identifying connected edges to v
                connected_edges = cellmap.edge_df[
                    (cellmap.edge_df["srce"] == v) | (cellmap.edge_df["trgt"] == v)
                ]
                edge_sum = 0
                for edge_idx, edge in connected_edges.iterrows():
                    length = connected_edges.loc[edge_idx, 'length']
                    edge_sum = edge_sum + length
               
                
                if edge_sum > edge_sum_threshold:
                    vertices_to_divide.append(v)
            
            remodel_nverts_stats.append(len(vertices_to_divide))
            print(f"  Vertices above threshold: {len(vertices_to_divide)}")

            if len(vertices_to_divide) > 0:
                divided_count = 0
                
                for v in vertices_to_divide:
                    try:
                        cellmap, parent_vert, new_vert, new_edge, opp_edge = auxFunctionsExpansion.split_vertex_expansion(
                            cellmap, v, geom, energyContributions_model, division_distance, retry_attempts=3
                        )
                        
                        if new_vert is not None:
                            # Recording lineage
                            parent_id = int(cellmap.vert_df.loc[parent_vert, "new_vert_id"])
                            cellmap.vert_df.loc[parent_vert, "divided_step"] = current_step
                            cellmap.vert_df.loc[new_vert, "new_vert_id"] = next_new_vert_id
                            cellmap.vert_df.loc[new_vert, "parent_vert_id"] = parent_id
                            cellmap.vert_df.loc[new_vert, "birth_step"] = current_step
                            next_new_vert_id += 1
                            divided_count += 1
                            
                    except Exception as e:
                        print(f"  Warning: division failed for vertex {v}: {e}")
                
                current_divisions = divided_count

                # Saving post-division checkpoint
                with open(os.path.join(output_dir, f"checkpoint_{current_step:06d}_postdiv.pkl"), "wb") as f:
                    pickle.dump(cellmap, f)

            print(f"  Divisions: {current_divisions}")

            if relax_after_division > 0:
                cellmap = relax(cellmap, relax_after_division)
                cellmap.reset_index()
                cellmap.reset_topo()

        # Relaxing model post apoptosis and divisions
        used_steps = 0
        if enable_collapses:
            used_steps += relax_after_collapse
        if enable_divisions:
            used_steps += relax_after_division

        remaining_steps = steps_per_cycle - used_steps
        if remaining_steps < 0:
            raise ValueError("relax_after_collapse + relax_after_division exceeds steps_per_cycle")

        if remaining_steps > 0:
            cellmap = relax(cellmap, remaining_steps)

        # Recording statistics
        division_stats.append(current_divisions)
        collapse_stats.append(current_collapses)

        # Sasving checkpoint and visualisation
        with open(os.path.join(output_dir, f"checkpoint_{current_step:06d}.pkl"), "wb") as f:
            pickle.dump(cellmap, f)

        fig, ax = auxFunctions.view(cellmap, geom, show_axes=True, xlim=xlim, ylim=ylim)
        title = f"Step {current_step} | {simulation_desc} | Div:{current_divisions} Col:{current_collapses}"
        plt.title(title, fontsize=10)
        plt.savefig(os.path.join(output_dir, f"progress_{current_step:06d}.png"), dpi=150, bbox_inches="tight")
        plt.close(fig)

        # Updating JSON statistics
        with open(json_path, "r") as f:
            existing_data = json.load(f)

        existing_data["last_saved_step"] = current_step
        existing_data["statistics"] = {
            "division_stats": division_stats if enable_divisions else None,
            "collapse_stats": collapse_stats if enable_collapses else None,
            "remodel_nverts_stats": remodel_nverts_stats,
            "collapse_target_stats": collapse_target_stats if enable_collapses else None,
            "inside_edges_total_stats": inside_edges_total_stats if enable_collapses else None,
        }

        with open(json_path, "w") as f:
            json.dump(existing_data, f, indent=2)

    return cellmap, division_stats, collapse_stats

In [ ]:
# Initializing cellmap, geometry, and energy contributions model
cellmap_init, geom, energyContributions_model = vertexModel2.initialize(40)
cellmap_init = MechanicalParams2.update(cellmap_init)

boundary_edges, boundary_faces, inside_edges, outside_edges, inside_faces, inside_vertices, outside_vertices = auxFunctions.identify_boundary_layers(cellmap_init, 1)

# Changing outside vertices mechanics
high_viscosity_value = 1000000  
for vertex_id in outside_vertices:
    if vertex_id in cellmap_init.vert_df.index:
        cellmap_init.vert_df.at[vertex_id, "viscosity"] = high_viscosity_value

        
# Allowing model to relax
energyContributions_model.compute_energy(cellmap_init)
[cellmap_init, geom, energyContributions_model, history_new, solver1] = vertexModel2.solveEuler(cellmap_init, geom, energyContributions_model, 200)


In [ ]:
cellmap, division_stats, collapse_stats = run_expansion_simulation(
    cellmap_init,
    geom,
    energyContributions_model,
    # Features to include in expansion
    enable_detachment=True,
    enable_divisions=True,
    enable_collapses=True,
    # Directory for saving 
    output_dir='Expansion_simulation', ### rename for different expansion conditions
    # How long to expand the tissue for
    total_steps=15000,
    steps_per_cycle=500,
    # Selecting amount of apoptosis
    collapse_fraction_per_remodel=0.0005,
    relax_after_collapse=10,
    # Selecting threshold for division
    edge_sum_threshold=7,
    division_distance=0.01,
    relax_after_division=10,
    # Selecting line tension relaxation
    tension_decrease_factor=0.4,
    # Selecting the fraction of edges to relax
    relax_tension_fraction=1,
    # Selecting factors for change in prefered area and area elasticity 
    pressure_increase_factor=8,
    area_elasticity_value=25,
    # Specifying capsule remodelling to allow expansion
    capsule_tension=300,
    capsule_viscosity=10000,
    # Limit cell cycle events for the inside of the tissue, to avoid boundary effects
    boundary_layers=5,
    # Plotting
    xlim=(-30, 70),
    ylim=(-30, 70),
)